In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from delta.tables import DeltaTable

# Slowly Changing Dimension - Initial and Incremental

In [0]:
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("/FileStore/rawcsv")

In [0]:
df = df.select("p_id","p_name","p_category").filter(col("p_id").isNotNull())

In [0]:
df.display()

p_id,p_name,p_category
1,cookies,food
2,almonds,food
3,Toothpase,merchandise
4,earphones,electronics
5,oil,merchandise
6,shirt,merchandise


In [0]:
initial_run = 0

In [0]:
if (initial_run == 0):
    delta_table = DeltaTable.forPath(spark,"/FileStore/rawcsvsink2")

    delta_table.alias("trg").merge(df.alias("src"), "trg.p_id = src.p_id")\
                            .whenMatchedUpdateAll()\
                            .whenNotMatchedInsertAll()\
                            .execute()

else:
    df.write.format("delta")\
            .mode("append")\
            .option("path","/FileStore/rawcsvsink2")\
            .saveAsTable("productsDim")

In [0]:
%sql
SELECT * FROM productsdim

p_id,p_name,p_category
1,cookies,food
2,almonds,food
3,Toothpase,merchandise
4,earphones,electronics
5,oil,merchandise
6,shirt,merchandise
